In [1]:
import pandas as pd

In [4]:
# ── 1. Load the three Excel files ──────────────────────────────────────────
# Update these paths to match your actual file names
plant_address  = pd.read_excel("Plant vs address.XLSX")   # Plant + address columns
plant_desc     = pd.read_excel("Plant Vs Plant Desc.XLSX")       # Plant + Plant Description
po_plant       = pd.read_excel("PO Vs Plant.XLSX")         # PO_NUMBER + Item + Plant

In [7]:
def normalise_plant(df):
    df["Plant"] = (
        df["Plant"]
        .astype(float)
        .round(0)
        .astype("Int64")
        .astype(str)
        .str.replace("<NA>", "", regex=False)
        .str.strip()
    )
    return df

plant_address  = normalise_plant(plant_address)
plant_desc     = normalise_plant(plant_desc)
po_plant       = normalise_plant(po_plant)

# ── Diagnose before merging ──────────────────────────────────────────────────
print("PO rows:          ", len(po_plant))
print("plant_desc dupes: ", plant_desc["Plant"].duplicated().sum())
print("plant_addr dupes: ", plant_address["Plant"].duplicated().sum())

# ── Deduplicate lookup tables on Plant (keep first occurrence) ───────────────
plant_desc    = plant_desc.drop_duplicates(subset="Plant", keep="first")
plant_address = plant_address.drop_duplicates(subset="Plant", keep="first")

print("After dedup — desc rows:", len(plant_desc), "| address rows:", len(plant_address))

# ── Merge ────────────────────────────────────────────────────────────────────
merged = po_plant.merge(plant_desc,    on="Plant", how="left")
merged = merged.merge(plant_address,   on="Plant", how="left")

print("Merged rows:", len(merged))   # should equal PO rows

# ── Reorder columns ──────────────────────────────────────────────────────────
po_cols      = ["PO_NUMBER", "Item", "Plant"]
desc_cols    = [c for c in plant_desc.columns    if c != "Plant"]
address_cols = [c for c in plant_address.columns if c != "Plant"]
merged = merged[po_cols + desc_cols + address_cols]

merged.to_excel("merged_plant_data.xlsx", index=False)
print(f"Done → merged_plant_data.xlsx")
print(f"Rows with no Plant Description match: {merged['Plant Description'].isna().sum()}")

PO rows:           56102
plant_desc dupes:  13400
plant_addr dupes:  0
After dedup — desc rows: 489 | address rows: 489
Merged rows: 56102
Done → merged_plant_data.xlsx
Rows with no Plant Description match: 4552


In [8]:


CITY_TO_STATE = {
    # Maharashtra
    "MUMBAI": "Maharashtra", "Mumbai": "Maharashtra", "NAVI MUMBAI": "Maharashtra",
    "Navi Mumbai": "Maharashtra", "PUNE": "Maharashtra", "Pune": "Maharashtra",
    "NASHIK": "Maharashtra", "NASIK": "Maharashtra", "AURANGABAD": "Maharashtra",
    "Aurangabad.": "Maharashtra", "NAGPUR": "Maharashtra", "Nagpur": "Maharashtra",
    "SHOLAPUR": "Maharashtra", "SOLAPUR": "Maharashtra", "Solapur": "Maharashtra",
    "SANGLI": "Maharashtra", "Sangli": "Maharashtra",
    "RAIGAD": "Maharashtra", "SILVASA": "Maharashtra",  # UT but near MH
    "CHANDRAPUR": "Maharashtra", "SINNAR": "Maharashtra", "MANMAD": "Maharashtra",
    "PAKNI": "Maharashtra", "GAIGOAN": "Maharashtra", "ALIBAG": "Maharashtra",
    "TALEGON": "Maharashtra", "Talegon": "Maharashtra", "THANE": "Maharashtra",
    "KOLHAPUR": "Maharashtra", "JALGAON": "Maharashtra",
    "KHOPOLI": "Maharashtra", "Khopoli": "Maharashtra",
    "URAN": "Maharashtra", "NEEMGAON": "Maharashtra",
    "SHIRDI": "Maharashtra", "Vashi": "Maharashtra", "VASHI": "Maharashtra",
    "PALGHAR": "Maharashtra",

    # Andhra Pradesh / Telangana
    "HYDERABAD": "Telangana", "SECUNDERABAD": "Telangana",
    "Secunderabad": "Telangana", "RAMAGUNDAM": "Telangana",
    "WARANGAL": "Telangana", "Warangal": "Telangana",
    "NIZAMABAD": "Telangana", "SURYAPET": "Telangana", "Suryapet": "Telangana",
    "BOGARAM": "Telangana", "Ghatkesar": "Telangana", "GHATKESAR": "Telangana",
    "KARIMNAGAR": "Telangana",
    "VISAKHAPATNAM": "Andhra Pradesh", "Visakhapatnam": "Andhra Pradesh",
    "VISAKH": "Andhra Pradesh", "VIJAYAWADA": "Andhra Pradesh",
    "Vijayawada": "Andhra Pradesh", "RAJAHMUNDRY": "Andhra Pradesh",
    "Rajahmundry": "Andhra Pradesh", "KADAPA": "Andhra Pradesh",
    "NELLORE": "Andhra Pradesh", "ANANTAPUR": "Andhra Pradesh",
    "TIRUPATI": "Andhra Pradesh", "ELURU": "Andhra Pradesh",
    "KAKINADA": "Andhra Pradesh", "DONAKONDA": "Andhra Pradesh",
    "BOGHAPURAM": "Andhra Pradesh", "KALAKADA": "Andhra Pradesh",
    "TADEPALLI": "Andhra Pradesh", "Tadepalli": "Andhra Pradesh",
    "BELLARY": "Karnataka",  # Note: Bellary is in Karnataka

    # Karnataka
    "BENGALURU": "Karnataka", "Bengaluru": "Karnataka",
    "BANGALORE": "Karnataka", "Bangalore": "Karnataka",
    "BANGALURU": "Karnataka", "COCHIN": "Kerala",  # overridden below
    "MANGALORE": "Karnataka", "GULBARGA": "Karnataka",
    "DHARWAR": "Karnataka", "HUBLI": "Karnataka",
    "Hassan": "Karnataka", "HASSAN": "Karnataka",
    "MYSORE": "Karnataka", "MYSURU": "Karnataka",
    "BELGAUM": "Karnataka", "Belgaum": "Karnataka", "BELAGAVI": "Karnataka",
    "TUMKUR": "Karnataka", "TUMKUR(DISTRICT)": "Karnataka",
    "NANDUR, GULBARGA": "Karnataka",

    # Kerala
    "COCHIN": "Kerala", "ERNAKULAM": "Kerala", "Ernakulam": "Kerala",
    "KOZHIKODE": "Kerala", "Kozhicode": "Kerala", "CALICUT": "Kerala",
    "PALGHAT": "Kerala", "KOCHI": "Kerala", "THRISSUR": "Kerala",
    "THIRUVANANTHAPURAM": "Kerala",

    # Tamil Nadu
    "CHENNAI": "Tamil Nadu", "Chennai": "Tamil Nadu",
    "COIMBATORE": "Tamil Nadu", "Coimbatore": "Tamil Nadu",
    "MADURAI": "Tamil Nadu", "TIRUNELVELI": "Tamil Nadu",
    "SALEM": "Tamil Nadu", "Trichy": "Tamil Nadu", "TRICHY": "Tamil Nadu",
    "DHARMAPURI": "Tamil Nadu", "THIRUVALLUVAR": "Tamil Nadu",
    "THANJAVUR": "Tamil Nadu", "TIRUPPUR": "Tamil Nadu",
    "VELLORE": "Tamil Nadu",

    # Rajasthan
    "JAIPUR": "Rajasthan", "AJMER": "Rajasthan", "JODHPUR": "Rajasthan",
    "Jodhpur": "Rajasthan", "KOTA": "Rajasthan", "UDAIPUR": "Rajasthan",
    "Udaipur": "Rajasthan", "BHARATPUR": "Rajasthan", "Bharatpur": "Rajasthan",
    "SALAWAS": "Rajasthan", "BARMER": "Rajasthan", "BIKANER": "Rajasthan",
    "AJMER": "Rajasthan", "TONK": "Rajasthan", "SIROHI": "Rajasthan",
    "SAMBHRA": "Rajasthan", "SURATGARH": "Rajasthan",
    "BALOTRA": "Rajasthan", "NALIYA": "Rajasthan",  # Naliya is Gujarat

    # Gujarat
    "AHMEDABAD": "Gujarat", "Ahmedabad": "Gujarat", "VADODARA": "Gujarat",
    "Vadodara.": "Gujarat", "SURAT": "Gujarat", "Hazira": "Gujarat",
    "RAJKOT": "Gujarat", "Rajkot": "Gujarat", "GANDHIDAM": "Gujarat",
    "GANDHINAGAR": "Gujarat", "PALANPUR": "Gujarat", "MUNDRA": "Gujarat",
    "Mundra": "Gujarat", "PIPAVAV": "Gujarat", "BHUJ": "Gujarat",
    "KUTCH": "Gujarat", "SILVASA": "Dadra & NH",
    "Vasco-Da-Gama": "Goa", "VASCO-DA-GAMA": "Goa", "PONDA": "Goa",
    "GOA": "Goa", "PANAJI": "Goa",

    # Madhya Pradesh
    "BHOPAL": "Madhya Pradesh", "Bhopal": "Madhya Pradesh",
    "JABALPUR": "Madhya Pradesh", "Jabalpur": "Madhya Pradesh",
    "GWALIOR": "Madhya Pradesh", "INDORE": "Madhya Pradesh",
    "SAGAR": "Madhya Pradesh", "MANGLIA": "Madhya Pradesh",
    "Manglia": "Madhya Pradesh", "MANDIR HASAUD": "Chhattisgarh",
    "RATLAM": "Madhya Pradesh", "KURAWAR": "Madhya Pradesh",
    "NEEMGAON": "Madhya Pradesh", "VIJAPUR": "Madhya Pradesh",

    # Chhattisgarh
    "RAIPUR": "Chhattisgarh", "Raipur": "Chhattisgarh",
    "BILASPUR": "Chhattisgarh", "SAMBALPUR": "Odisha",  # override below
    "MANDIR HASAUD": "Chhattisgarh",

    # Uttar Pradesh
    "LUCKNOW": "Uttar Pradesh", "Lucknow": "Uttar Pradesh",
    "KANPUR": "Uttar Pradesh", "Kanpur": "Uttar Pradesh",
    "KANPUR DEHAT": "Uttar Pradesh", "MATHURA": "Uttar Pradesh",
    "Mathura": "Uttar Pradesh", "AGRA": "Uttar Pradesh", "Agra": "Uttar Pradesh",
    "MEERUT": "Uttar Pradesh", "VARANASI": "Uttar Pradesh",
    "PRAYAGRAJ": "Uttar Pradesh", "BAREILLY": "Uttar Pradesh",
    "GORAKHPUR": "Uttar Pradesh", "UNNAO": "Uttar Pradesh",
    "AONLA": "Uttar Pradesh", "MUGHALSARAI": "Uttar Pradesh",
    "BAITALPUR": "Uttar Pradesh", "BAHADURGARH": "Haryana",  # override
    "JHANSI": "Uttar Pradesh", "GHAZIABAD": "Uttar Pradesh",
    "SHAHJAHANPUR": "Uttar Pradesh", "MAINPURI": "Uttar Pradesh",
    "FARRUKHABAD": "Uttar Pradesh", "BADAUN": "Uttar Pradesh",
    "MOTIHARI": "Bihar",  # override
    "BAITALPUR": "Uttar Pradesh", "LUCKNOW": "Uttar Pradesh",
    "FAIZABAD": "Uttar Pradesh", "GONDA": "Uttar Pradesh",
    "UNNAO": "Uttar Pradesh", "ARAH": "Bihar",  # override

    # Bihar
    "PATNA": "Bihar", "Patna": "Bihar", "BEGUSARAI": "Bihar",
    "Begusarai": "Bihar", "BOKARO": "Jharkhand",
    "MOTIHARI": "Bihar", "PURNEA": "Bihar", "MUZAFFARPUR": "Bihar",
    "ARAH": "Bihar", "DUMKA": "Jharkhand", "DHANBAD": "Jharkhand",
    "JAMSHEDPUR": "Jharkhand", "Ranchi": "Jharkhand", "RANCHI": "Jharkhand",
    "SOMNATHPUR": "Odisha", "PARADEEP": "Odisha",

    # Haryana
    "HISAR": "Haryana", "HISSAR": "Haryana", "REWARI": "Haryana",
    "Rewari": "Haryana", "BAHADURGARH": "Haryana",
    "GURGAON": "Haryana", "SONIPAT": "Haryana", "PANIPAT": "Haryana",
    "Panipat": "Haryana", "JIND": "Haryana", "ROHTAK": "Haryana",
    "FARIDABAD": "Haryana",

    # Punjab
    "BATHINDA": "Punjab", "BHATINDA": "Punjab", "Bhatinda": "Punjab",
    "JALANDHAR": "Punjab", "SANGRUR": "Punjab", "NALAGARH": "Himachal Pradesh",
    "AMRITSAR": "Punjab", "MOHALI": "Punjab", "CHANDIGARH": "Chandigarh UT",

    # Himachal Pradesh
    "SHIMLA": "Himachal Pradesh", "NALAGARH": "Himachal Pradesh",
    "KANGRA": "Himachal Pradesh", "HOSHIARPUR": "Punjab",  # actually Punjab

    # J&K
    "JAMMU": "J&K", "PAMPORE": "J&K", "LADAKH": "Ladakh",
    "AWANTIPUR": "J&K", "BUDGAM": "J&K", "UDHAMPUR": "J&K",
    "UDHAM SINGH NAGAR": "Uttarakhand",

    # Uttarakhand
    "ROORKEE": "Uttarakhand", "DEHRADUN": "Uttarakhand",
    "Dehradun": "Uttarakhand", "HALDWANI": "Uttarakhand",
    "UDHAM SINGH NAGAR": "Uttarakhand", "HARIDWAR": "Uttarakhand",

    # Delhi
    "DELHI": "Delhi", "NEW DELHI": "Delhi", "New Delhi": "Delhi",

    # West Bengal
    "KOLKATA": "West Bengal", "Kolkata": "West Bengal",
    "CALCUTTA": "West Bengal", "BUDGE BUDGE": "West Bengal",
    "HALDIA": "West Bengal", "BURDWAN": "West Bengal",
    "Durgapur": "West Bengal", "DURGAPUR": "West Bengal",
    "SILIGURI": "West Bengal", "MALDA": "West Bengal",
    "PANAGARH": "West Bengal", "PURULIA": "West Bengal",
    "PURBA MEDINIPUR": "West Bengal", "PURBA BARDHAMAN": "West Bengal",
    "BARHI": "Jharkhand",

    # Odisha
    "BHUBANESWAR": "Odisha", "Bhubaneswar": "Odisha",
    "SAMBALPUR": "Odisha", "PARADEEP": "Odisha", "RAYAGADA": "Odisha",
    "JATNI": "Odisha", "SOMNATHPUR": "Odisha", "ROURKELA": "Odisha",

    # Assam / NE
    "GUWAHATI": "Assam", "Guwahati": "Assam",
    "NUMALIGARH": "Assam", "BONGAIGAON": "Assam", "GOALPARA": "Assam",
    "TINSUKIA": "Assam", "KAMAVARUPUKOTA": "Assam",
    "NIULAND": "Nagaland", "SAJGAON": "Assam",

    # Odisha more
    "BOKARO": "Jharkhand",
}


In [10]:
# Load your Excel file
df = pd.read_excel("merged_plant_data.xlsx")

# Normalize city names for matching (strip whitespace, uppercase lookup)

df["State"] = df["City"].str.strip().map(
    lambda x: (
        CITY_TO_STATE.get(x)
        or CITY_TO_STATE.get(x.upper())
        or CITY_TO_STATE.get(x.title())
        or "Unknown"
    ) if isinstance(x, str) else "Unknown"
)

# Save back to Excel
df.to_excel("merged_plant_data.xlsx", index=False)
print(df[["City", "State"]].head(20))


             City           State
0     RAJAHMUNDRY  Andhra Pradesh
1         CHENNAI      Tamil Nadu
2   VISAKHAPATNAM  Andhra Pradesh
3            PUNE     Maharashtra
4         MANGLIA  Madhya Pradesh
5            PUNE     Maharashtra
6             NaN         Unknown
7             NaN         Unknown
8             NaN         Unknown
9         MANGLIA  Madhya Pradesh
10        MADURAI      Tamil Nadu
11            NaN         Unknown
12    NAVI MUMBAI     Maharashtra
13           PUNE     Maharashtra
14           PUNE     Maharashtra
15    NAVI MUMBAI     Maharashtra
16            NaN         Unknown
17            NaN         Unknown
18            NaN         Unknown
19    NAVI MUMBAI     Maharashtra


In [ ]:
import pandas as pd

In [2]:


# Load files
merged     = pd.read_excel("merged_plant_data.xlsx")
clustered  = pd.read_excel("Clustered_Output_final_testing_2.xlsx")

# Normalise PO_NUMBER (strip whitespace, cast to str)
for df in [merged, clustered]:
    df["PO_NUMBER"] = df["PO_NUMBER"].astype(str).str.strip()

# Diagnose before merging
print("merged rows:    ", len(merged))
print("clustered rows: ", len(clustered))
print("clustered PO dupes:", clustered["PO_NUMBER"].duplicated().sum())

# Merge — left join keeps all clustered rows
final = clustered.merge(merged, on="PO_NUMBER", how="left")

print("Final rows:", len(final))  # should equal clustered rows

final.to_excel("final_merged.xlsx", index=False)
print("Done → final_merged.xlsx")
print(f"Rows with no Plant match: {final['Plant_x'].isna().sum() if 'Plant_x' in final.columns else final['Plant'].isna().sum()}")

merged rows:     56102
clustered rows:  55279
clustered PO dupes: 0
Final rows: 55279
Done → final_merged.xlsx
Rows with no Plant match: 2614


In [3]:
df = pd.read_excel("final_merged.xlsx")
NULL_MARKERS = {"—", "-", "N/A", "NA", "na", "n/a", "", "NULL", "null", "None"}

def assign_slab(value):
    if pd.isna(value) or str(value).strip() in NULL_MARKERS:
        return None
    try:
        value = float(str(value).replace(",", "").strip())
    except ValueError:
        return None  # any other unexpected string → skip
    
    if value <= 2_000_000:
        return "0-20L"
    elif value <= 5_000_000:
        return "20-50L"
    elif value <= 10_000_000:
        return "50-100L"
    elif value <= 20_000_000:
        return "100-200L"
    elif value <= 30_000_000:
        return "200-300L"
    else:
        return ">300L"

for i, row in df.iterrows():
    slab = assign_slab(row["BASICVALUE"])
    df.at[i, "BASIC_VALUE_SLAB"] = slab
    print(f"Row {i} | BASICVALUE: {row['BASICVALUE']} → Slab: {slab}")


df.to_excel("CLUSTERED_PO_TITLES_55K.xlsx", index=False)
print("Done!")

Row 0 | BASICVALUE: 500000 → Slab: 0-20L
Row 1 | BASICVALUE: 750000 → Slab: 0-20L
Row 2 | BASICVALUE: 480000 → Slab: 0-20L
Row 3 | BASICVALUE: 28155400 → Slab: 200-300L
Row 4 | BASICVALUE: 28155400 → Slab: 200-300L
Row 5 | BASICVALUE: 28155400 → Slab: 200-300L
Row 6 | BASICVALUE: 28155400 → Slab: 200-300L
Row 7 | BASICVALUE: 28155400 → Slab: 200-300L
Row 8 | BASICVALUE: 35000000 → Slab: >300L
Row 9 | BASICVALUE: 35000000 → Slab: >300L
Row 10 | BASICVALUE: 16900000 → Slab: 100-200L
Row 11 | BASICVALUE: 16900000 → Slab: 100-200L
Row 12 | BASICVALUE: 16900000 → Slab: 100-200L
Row 13 | BASICVALUE: 16900000 → Slab: 100-200L
Row 14 | BASICVALUE: 16900000 → Slab: 100-200L
Row 15 | BASICVALUE: 16900000 → Slab: 100-200L
Row 16 | BASICVALUE: 16900000 → Slab: 100-200L
Row 17 | BASICVALUE: 16900000 → Slab: 100-200L
Row 18 | BASICVALUE: 9886016.63 → Slab: 50-100L
Row 19 | BASICVALUE: 21186440.68 → Slab: 200-300L
Row 20 | BASICVALUE: 21186440.68 → Slab: 200-300L
Row 21 | BASICVALUE: 21186440.68 → Sl

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Load & sort ───────────────────────────────────────────────────────────────
df = pd.read_excel('CLUSTERED_PO_TITLES_55K.xlsx')

df['Category']     = df['Category'].fillna('Unmapped')
df['Sub Category'] = df['Sub Category'].fillna('Unmapped')
df['State']        = df['State'].fillna('Unknown')
df['BASICVALUE']   = pd.to_numeric(df['BASICVALUE'], errors='coerce').fillna(0)

df = df.sort_values(
    ['Category', 'Sub Category', 'State', 'BASICVALUE'],
    ascending=[True, True, True, True]
).reset_index(drop=True)

# ── Colour palettes (cat header, subcat body) ─────────────────────────────────
CATEGORY_PALETTES = [
    ("1F4E79", "BDD7EE"),
    ("375623", "E2EFDA"),
]
STATE_TINTS = ["F0F0F0", "FAFAFA"]

def make_fill(h): return PatternFill("solid", start_color=h, fgColor=h)
def make_font(bold=False, color="000000", size=9):
    return Font(name="Arial", bold=bold, color=color, size=size)

thin = Side(style="thin", color="CCCCCC")
med  = Side(style="medium", color="555555")
thin_border = Border(left=thin, right=thin, top=thin, bottom=thin)

# ── Workbook ──────────────────────────────────────────────────────────────────
wb = Workbook()
ws = wb.active
ws.title = "Procurement Data"

COLS = ['Cluster ID','Standard Name','Category','Sub Category','PO_TITLE',
        'BASICVALUE','BASIC_VALUE_SLAB','PO_NUMBER','VENDOR','VENDOR_NAME',
        'Plant','Plant Description','City','State','isTendered']

# Header
ws.row_dimensions[1].height = 28
hfill = make_fill("1C2833")
hfont = make_font(bold=True, color="FFFFFF", size=9)
for ci, col in enumerate(COLS, 1):
    c = ws.cell(row=1, column=ci, value=col)
    c.fill = hfill; c.font = hfont
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.border = thin_border

categories  = df['Category'].unique()
cat_palette = {cat: CATEGORY_PALETTES[i % len(CATEGORY_PALETTES)]
               for i, cat in enumerate(categories)}

current_cat = current_subcat = current_state = None
state_toggle = 0
left  = Alignment(horizontal="left",  vertical="center")
right = Alignment(horizontal="right", vertical="center")

for row_num, (_, rec) in enumerate(df.iterrows(), start=2):
    cat    = rec['Category']
    subcat = rec['Sub Category']
    state  = rec['State']
    cat_hex, subcat_hex = cat_palette[cat]

    # Track state toggle per subcat
    if subcat != current_subcat:
        current_subcat = subcat
        current_state  = state
        state_toggle   = 0
    elif state != current_state:
        current_state = state
        state_toggle  = 1 - state_toggle

    is_new_cat = (cat != current_cat)
    if is_new_cat:
        current_cat = cat

    ws.row_dimensions[row_num].height = 15

    for ci, col in enumerate(COLS, 1):
        val  = rec.get(col, "")
        cell = ws.cell(row=row_num, column=ci, value=val)
        cell.border = thin_border

        # Colour logic
        if is_new_cat:
            # first row of a category block → dark category colour
            cell.fill = make_fill(cat_hex)
            cell.font = make_font(bold=True, color="FFFFFF", size=9)
        elif ci <= 4:
            # identity columns → subcat colour
            cell.fill = make_fill(subcat_hex)
            cell.font = make_font(bold=False, color="1C2833", size=9)
        else:
            # data columns → state zebra
            cell.fill = make_fill(STATE_TINTS[state_toggle])
            cell.font = make_font(bold=False, color="1C2833", size=9)

        cell.alignment = right if col == 'BASICVALUE' else left
        if col == 'BASICVALUE':
            cell.number_format = '#,##0.00'

# Thick bottom border at category boundaries
for idx in range(len(df) - 1):
    if df.loc[idx, 'Category'] != df.loc[idx+1, 'Category']:
        er = idx + 2
        for ci in range(1, len(COLS)+1):
            ws.cell(er, ci).border = Border(left=thin, right=thin, top=thin, bottom=med)

# Column widths
col_widths = {
    'Cluster ID':14,'Standard Name':30,'Category':24,'Sub Category':28,
    'PO_TITLE':45,'BASICVALUE':14,'BASIC_VALUE_SLAB':16,'PO_NUMBER':14,
    'VENDOR':12,'VENDOR_NAME':28,'Plant':8,'Plant Description':22,
    'City':14,'State':16,'isTendered':11
}
for ci, col in enumerate(COLS, 1):
    ws.column_dimensions[get_column_letter(ci)].width = col_widths.get(col, 15)

ws.freeze_panes = "A2"

# ── Legend sheet ──────────────────────────────────────────────────────────────
leg = wb.create_sheet("Legend")
leg.column_dimensions['A'].width = 32
leg.column_dimensions['B'].width = 35

for ci,txt in enumerate(["Category","Colour"],1):
    c = leg.cell(1, ci, txt)
    c.fill = make_fill("1C2833"); c.font = make_font(bold=True, color="FFFFFF", size=10)
    c.border = thin_border; c.alignment = Alignment(horizontal="center")

for i, cat in enumerate(categories, 2):
    ch, sh = cat_palette[cat]
    c1 = leg.cell(i, 1, cat)
    c1.fill = make_fill(ch); c1.font = make_font(bold=True, color="FFFFFF", size=9)
    c1.border = thin_border
    c2 = leg.cell(i, 2, "Sub-category / data rows")
    c2.fill = make_fill(sh); c2.font = make_font(size=9, color="1C2833")
    c2.border = thin_border

r = len(categories) + 3
leg.cell(r, 1, "State alternation (data cols):").font = make_font(bold=True, size=9)
for j, (label, tint) in enumerate([("State group A", STATE_TINTS[0]),("State group B", STATE_TINTS[1])], r+1):
    leg.cell(j,1,label).fill = make_fill(tint); leg.cell(j,1).border = thin_border
    leg.cell(j,1).font = make_font(size=9)
    leg.cell(j,2,"Alternating per state within each Sub Category").font = make_font(size=9)
    leg.cell(j,2).border = thin_border



FileNotFoundError: [Errno 2] No such file or directory: '/mnt/user-data/outputs/CLUSTERED_PO_STYLED.xlsx'

In [5]:
# ── Save ──────────────────────────────────────────────────────────────────────
out = "CLUSTERED_PO_STYLED.xlsx"
wb.save(out)
print(f"Done → {out}  |  data rows: {len(df)}")

Done → CLUSTERED_PO_STYLED.xlsx  |  data rows: 55279
